## 3. Feature Engineering & Preprocessing

### Step 3.1: Encoding Categorical Variables

In [68]:
# Perform One-Hot Encoding on the 'tenure' and 'propertyType' columns
df_clean4 = pd.get_dummies(df_clean3, columns=['tenure', 'propertyType'], drop_first=True)

# Check the first few rows to see the result
print(df_clean4.head())

   postcode   latitude  longitude  bathrooms  bedrooms  floorAreaSqM  \
0  EC4A 1JQ  51.517282  -0.110314        1.0       1.0          45.0   
1  EC4A 1JQ  51.517282  -0.110314        1.0       0.0          84.0   
2  SW1P 2TA  51.495505  -0.132379        2.0       2.0          71.0   
3   SE5 7HS  51.478185  -0.092201        1.0       1.0          64.0   
4   N10 3RL  51.588774  -0.139599        1.0       4.0         137.0   

   livingRooms  rentEstimate_lowerPrice  rentEstimate_currentPrice  \
0          1.0                   2100.0                     2350.0   
1          0.0                   2100.0                     2350.0   
2          1.0                   2650.0                     2950.0   
3          1.0                   1850.0                     2000.0   
4          2.0                   4350.0                     4850.0   

   rentEstimate_upperPrice  ...  propertyType_Mid Terrace Bungalow  \
0                   2600.0  ...                              False   
1     

### 3.2 Scaling Numeric Features:

In [69]:
# List of numeric columns to scale
numeric_columns = ['floorAreaSqM', 'bathrooms', 'bedrooms', 'livingRooms', 'rentEstimate_currentPrice']

# Initialize StandardScaler
scaler = StandardScaler()

# Scale numeric features
df_clean4[numeric_columns] = scaler.fit_transform(df_clean4[numeric_columns])

print(df_clean4[numeric_columns].head())

   floorAreaSqM  bathrooms  bedrooms  livingRooms  rentEstimate_currentPrice
0     -0.975782  -0.534724 -0.963843    -0.241637                  -0.369174
1     -0.258116  -0.534724 -1.731006    -1.798594                  -0.369174
2     -0.497338   0.944552 -0.196679    -0.241637                  -0.196819
3     -0.626150  -0.534724 -0.963843    -0.241637                  -0.469715
4      0.717174  -0.534724  1.337648     1.315320                   0.348974


### 3.3 Create New Features:

In [70]:
# Create a new column for price per square meter
df_clean4['price_per_sqm'] = df_clean4['rentEstimate_currentPrice'] / df_clean4['floorAreaSqM']

print(df_clean4[['rentEstimate_currentPrice', 'floorAreaSqM', 'price_per_sqm']].head())

   rentEstimate_currentPrice  floorAreaSqM  price_per_sqm
0                  -0.369174     -0.975782       0.378337
1                  -0.369174     -0.258116       1.430267
2                  -0.196819     -0.497338       0.395745
3                  -0.469715     -0.626150       0.750164
4                   0.348974      0.717174       0.486596


To calculate the Distance from City Centre using latitude and longitude, we can use the Haversine formula.

We'll need the latitude and longitude of Central London (from Google: latitude = 51.5074, longitude = -0.1278).

In [71]:
# Function to calculate the distance using Haversine formula
def haversine(lat1, lon1, lat2, lon2):
    # Convert degrees to radians
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])

    # Haversine formula
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

    # Earth's radius in kilometers
    R = 6371.0
    distance = R * c  # distance in kilometers
    return distance

# Coordinates for Central London
central_london_lat = 51.5074
central_london_lon = -0.1278

# Calculate the distance for each property in the dataset
df_clean4['distance_from_centre'] = df_clean4.apply(
    lambda row: haversine(row['latitude'], row['longitude'], central_london_lat, central_london_lon),
    axis=1
)

print(df_clean4[['latitude', 'longitude', 'distance_from_centre']].head())


    latitude  longitude  distance_from_centre
0  51.517282  -0.110314              1.634534
1  51.517282  -0.110314              1.634534
2  51.495505  -0.132379              1.360143
3  51.478185  -0.092201              4.077599
4  51.588774  -0.139599              9.085086
